# backward-on-scalar-loss — faded example 2: Supply the gradient= argument for non-scalar backward()

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`. Running the beacon reports progress on the `PyTorch: backward()` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: backward()` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-on-scalar-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-on-scalar-loss"
DD_SUBTOPIC = "PyTorch: backward()"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Calling `.backward()` on a non-scalar tensor `y` raises a RuntimeError unless you pass `gradient=v`. With `v = ones_like(y)` the result is `x.grad = J^T @ 1`, identical to `y.sum().backward()`. The vector `v` is the upstream (VJP) seed.

## Faded exercise 2

### Faded — build the VJP seed vector

Implement `faded_vjp(x, f)`. A fresh leaf and the forward pass `y = f(x_leaf)` are given. **Complete the missing step: construct the seed `v` of ones with the same shape as `y`** (using `t.ones_like`) so that `y.backward(gradient=v)` computes `d(sum(f(x)))/dx`. Return `x_leaf.grad`.

**Fill in:** Construct the all-ones VJP seed vector with the same shape and dtype as y.

In [ ]:
def faded_vjp(x, f):
    x_leaf = x.detach().clone().requires_grad_(True)
    y = f(x_leaf)
    v = t.ones_like(y)
    y.backward(gradient=v)
    return x_leaf.grad.detach().clone()

t.manual_seed(0)
x = t.tensor([1.0, 2.0, 3.0])
g = faded_vjp(x, lambda z: z ** 2)
print(g)


def _test():
    x = t.tensor([1.0, 2.0, 3.0])
    g = faded_vjp(x, lambda z: z ** 2)
    # d(sum(z^2))/dz = 2z
    assert t.allclose(g, 2 * x), 'VJP for z**2 should be 2*x'
    # cross-check against sum().backward()
    xl = x.detach().clone().requires_grad_(True)
    (xl ** 2).sum().backward()
    assert t.allclose(g, xl.grad), 'must match sum().backward()'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def faded_vjp(x, f):
    x_leaf = x.detach().clone().requires_grad_(True)
    y = f(x_leaf)
    v = t.ones_like(y)
    y.backward(gradient=v)
    return x_leaf.grad.detach().clone()

t.manual_seed(0)
x = t.tensor([1.0, 2.0, 3.0])
g = faded_vjp(x, lambda z: z ** 2)
print(g)
```
</details>